# AIRON Agent Passport Pilot Notebook

This Google Colab-ready notebook is the Phase 1 external pilot harness for the AIRON Agent Passport.

It proves the minimum TRL-5 AIP flow against a deployed `aip-service` endpoint:

1. Connect to AIP service.
2. Issue or reuse an AIP token.
3. Issue an `AgentPassport`.
4. Verify the passport and expect `PASS`.
5. Tamper with the passport and expect `TOXIC`.
6. Attach passport metadata to common AI/ML workflow objects.
7. Export `pilot_results.json` and `agent.passport.json` as pilot evidence.


## Before You Run

You need a reachable AIP service URL. In Colab, set these values in the next cell:

- `AIP_BASE_URL`: public or tunneled URL for `aip-service`, for example `https://your-aip-service.example.com`.
- `AIP_TOKEN`: optional existing AIP token. If empty, the notebook calls `/v1/identity/issue` first.

For private pilots, run this notebook from a tester account and keep the generated `pilot_results.json` as evidence.


In [ ]:
# Optional: install the Python SDK directly from this branch.
# Keep INSTALL_SDK = False if the repo is private or network install is blocked.
INSTALL_SDK = False

if INSTALL_SDK:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/richardrich999888-rgb/AIP-agent-identity-protocol-.git@codex/val-ladder-sync#subdirectory=python/airon-aip-sdk",
    ])


In [ ]:
import copy
import hashlib
import json
import os
import platform
import time
import urllib.error
import urllib.request
import uuid
from pathlib import Path

# Configure these for the pilot environment.
AIP_BASE_URL = os.environ.get("AIP_BASE_URL", "http://127.0.0.1:8081")
AIP_TOKEN = os.environ.get("AIP_TOKEN", "")
PILOT_RESULTS_PATH = Path("pilot_results.json")
PASSPORT_PATH = Path("agent.passport.json")

print("AIP_BASE_URL:", AIP_BASE_URL)
print("AIP_TOKEN supplied:", bool(AIP_TOKEN))


In [ ]:
def request_json(method, path, body=None, headers=None, timeout=30):
    url = f"{AIP_BASE_URL.rstrip('/')}/{path.lstrip('/')}"
    payload = None
    request_headers = {"accept": "application/json"}
    if headers:
        request_headers.update(headers)
    if body is not None:
        payload = json.dumps(body).encode("utf-8")
        request_headers["content-type"] = "application/json"

    request = urllib.request.Request(url, data=payload, headers=request_headers, method=method)
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode("utf-8")
            elapsed_ms = (time.perf_counter() - started) * 1000
            return int(response.status), json.loads(raw or "{}"), elapsed_ms
    except urllib.error.HTTPError as exc:
        raw = exc.read().decode("utf-8")
        elapsed_ms = (time.perf_counter() - started) * 1000
        try:
            payload = json.loads(raw or "{}")
        except json.JSONDecodeError:
            payload = {"error": raw}
        return int(exc.code), payload, elapsed_ms


def require_success(status, payload, label):
    if status < 200 or status >= 300:
        raise RuntimeError(f"{label} failed with HTTP {status}: {payload}")
    return payload


def sha256_json(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


## 1. Service Health Check

This proves the notebook can reach the deployed AIP service before generating pilot evidence.


In [ ]:
status, health, health_ms = request_json("GET", "/health")
health = require_success(status, health, "health check")
print(json.dumps(health, indent=2))
print(f"health_latency_ms={health_ms:.2f}")


## 2. Issue or Reuse an AIP Token

If `AIP_TOKEN` is empty, this cell creates a fresh AI-model identity token through `/v1/identity/issue`.


In [ ]:
identity_issue = None
identity_issue_ms = None

if not AIP_TOKEN:
    identity_request = {
        "agent_id": str(uuid.uuid4()),
        "agent_type": "ai_model",
        "metadata": {
            "pilot": "airon-agent-passport-colab",
            "runtime": f"python-{platform.python_version()}",
            "environment": "google-colab" if "COLAB_RELEASE_TAG" in os.environ else "notebook",
        },
        "ttl_seconds": 3600,
    }
    status, identity_issue, identity_issue_ms = request_json("POST", "/v1/identity/issue", identity_request)
    identity_issue = require_success(status, identity_issue, "identity issue")
    AIP_TOKEN = identity_issue["token"]
    print("Issued AIP token for agent_id:", identity_request["agent_id"])
    print(f"identity_issue_latency_ms={identity_issue_ms:.2f}")
else:
    print("Using supplied AIP_TOKEN")

print("AIP_TOKEN prefix:", AIP_TOKEN[:24] + "...")


## 3. Issue an AgentPassport

This is the main pilot action: AIP validates the token, creates a passport body, signs it, and records a passport hash.


In [ ]:
passport_issue_request = {
    "aip_token": AIP_TOKEN,
    "friendly_name": "InvoiceBot-Colab-Pilot",
    "capabilities": ["invoice.read", "ledger.post"],
    "ttl_seconds": 3600,
    "max_delegation_depth": 3,
    "metadata": {
        "runtime": f"python-{platform.python_version()}",
        "framework": "pilot-notebook",
        "image_hash": "sha256:colab-notebook-runtime",
    },
    "environmental_lock": {
        "runtime": f"python-{platform.python_version()}",
        "image_hash": "sha256:colab-notebook-runtime",
    },
}

status, passport_issue, passport_issue_ms = request_json("POST", "/v1/passports/issue", passport_issue_request)
passport_issue = require_success(status, passport_issue, "passport issue")
passport = passport_issue["passport"]
passport_hash = passport_issue["passport_hash"]

PASSPORT_PATH.write_text(json.dumps(passport, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("passport_id:", passport["body"]["passport_id"])
print("agent_id:", passport["body"]["subject"]["agent_id"])
print("capabilities:", passport["body"]["constraints"]["capabilities"])
print("passport_hash:", passport_hash)
print(f"passport_issue_latency_ms={passport_issue_ms:.2f}")
print("saved:", PASSPORT_PATH)


## 4. Verify the Passport: Expected `PASS`

This confirms structural validity, temporal validity, and Ed25519 signature integrity.


In [ ]:
status, passport_verify, passport_verify_ms = request_json("POST", "/v1/passports/verify", {"passport": passport})
passport_verify = require_success(status, passport_verify, "passport verify")

print(json.dumps(passport_verify, indent=2))
print(f"passport_verify_latency_ms={passport_verify_ms:.2f}")

assert passport_verify["valid"] is True
assert passport_verify["status"] == "PASS"


## 5. Tamper Test: Expected `TOXIC`

This modifies the capability list after issuance. Verification must fail because the signature no longer matches the canonical passport body.


In [ ]:
tampered_passport = copy.deepcopy(passport)
tampered_passport["body"]["constraints"]["capabilities"].append("ledger.admin")

status, tampered_verify, tampered_verify_ms = request_json(
    "POST",
    "/v1/passports/verify",
    {"passport": tampered_passport},
)
tampered_verify = require_success(status, tampered_verify, "tampered passport verify")

print(json.dumps(tampered_verify, indent=2))
print(f"tampered_verify_latency_ms={tampered_verify_ms:.2f}")

assert tampered_verify["valid"] is False
assert tampered_verify["status"] == "TOXIC"


## 6. Optional Python SDK Flow

If `INSTALL_SDK = True` succeeded, this cell exercises the high-level `AgentPassport` SDK API. The direct HTTP pilot above is still the source of evidence.


In [ ]:
sdk_result = {"available": False, "status": "not_run"}

try:
    from airon_aip import AIPConfig, AgentPassport

    passport_service = AgentPassport(AIPConfig(base_url=AIP_BASE_URL, aip_token=AIP_TOKEN))
    sdk_passport = passport_service.issue(
        friendly_name="InvoiceBot-SDK-Pilot",
        capabilities=["invoice.read"],
        metadata={"runtime": f"python-{platform.python_version()}", "framework": "sdk"},
    )
    sdk_verification = passport_service.verify(sdk_passport)
    sdk_result = {
        "available": True,
        "status": sdk_verification.status,
        "is_valid": sdk_verification.is_valid,
        "passport_id": sdk_passport.passport_id,
        "passport_hash": sdk_passport.passport_hash,
    }
except Exception as exc:
    sdk_result = {"available": False, "status": "skipped", "reason": repr(exc)}

print(json.dumps(sdk_result, indent=2))


## 7. Attach Passport Metadata to AI/ML Workflow Objects

This creates lightweight integration evidence for common Python agent stacks without requiring all optional dependencies to be installed.


In [ ]:
passport_metadata = {
    "airon_passport_id": passport["body"]["passport_id"],
    "airon_agent_id": passport["body"]["subject"]["agent_id"],
    "airon_capabilities": passport["body"]["constraints"]["capabilities"],
    "airon_passport_hash": passport_hash,
}

integration_results = {}

try:
    import pandas as pd

    df = pd.DataFrame([{"invoice_id": "INV-001", "amount": 1250.00}])
    df.attrs["airon_agent_passport"] = passport_metadata
    integration_results["pandas"] = {"available": True, "attrs_present": "airon_agent_passport" in df.attrs}
except Exception as exc:
    integration_results["pandas"] = {"available": False, "reason": repr(exc)}

try:
    import numpy as np

    arr = np.array([1.0, 2.0, 3.0])
    arr_digest = hashlib.sha256(arr.tobytes()).hexdigest()
    integration_results["numpy"] = {"available": True, "array_sha256": arr_digest}
except Exception as exc:
    integration_results["numpy"] = {"available": False, "reason": repr(exc)}

try:
    import torch

    model = torch.nn.Linear(2, 1)
    model.airon_agent_passport = passport_metadata
    integration_results["torch"] = {"available": True, "metadata_present": hasattr(model, "airon_agent_passport")}
except Exception as exc:
    integration_results["torch"] = {"available": False, "reason": repr(exc)}

try:
    from langchain_core.runnables import RunnableLambda

    runnable = RunnableLambda(lambda value: value).with_config(metadata=passport_metadata)
    output = runnable.invoke({"invoice_id": "INV-001"})
    integration_results["langchain"] = {"available": True, "output": output}
except Exception as exc:
    integration_results["langchain"] = {"available": False, "reason": repr(exc)}

print(json.dumps(integration_results, indent=2, default=str))


## 8. Export Pilot Evidence

`pilot_results.json` is the artifact to attach to a TRL-5 readiness dossier or design-partner pilot report.


In [ ]:
pilot_results = {
    "pilot_name": "airon-agent-passport-colab",
    "created_at_unix": int(time.time()),
    "aip_base_url": AIP_BASE_URL,
    "service_health": health,
    "latency_ms": {
        "health": round(health_ms, 3),
        "identity_issue": None if identity_issue_ms is None else round(identity_issue_ms, 3),
        "passport_issue": round(passport_issue_ms, 3),
        "passport_verify_pass": round(passport_verify_ms, 3),
        "passport_verify_toxic": round(tampered_verify_ms, 3),
    },
    "passport": {
        "passport_id": passport["body"]["passport_id"],
        "agent_id": passport["body"]["subject"]["agent_id"],
        "capabilities": passport["body"]["constraints"]["capabilities"],
        "expires_at": passport["body"]["constraints"]["expires_at"],
        "passport_hash": passport_hash,
        "local_json_sha256": sha256_json(passport),
    },
    "verification": {
        "pass": passport_verify,
        "toxic_after_tamper": tampered_verify,
    },
    "sdk_result": sdk_result,
    "integration_results": integration_results,
    "trl_note": "Relevant-environment evidence when executed by a pilot tester against a deployed AIP service.",
}

PILOT_RESULTS_PATH.write_text(json.dumps(pilot_results, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("wrote:", PILOT_RESULTS_PATH)
print("wrote:", PASSPORT_PATH)
print(json.dumps(pilot_results, indent=2, default=str))


In [ ]:
# Colab convenience: download generated evidence files.
try:
    from google.colab import files

    files.download(str(PILOT_RESULTS_PATH))
    files.download(str(PASSPORT_PATH))
except Exception:
    print("Download helper unavailable. Files are available in the notebook runtime:")
    print(" -", PILOT_RESULTS_PATH.resolve())
    print(" -", PASSPORT_PATH.resolve())
